# Procedimiento

### Lectura del EEG crudo
Parámetros:
 - archivo_r2a: ruta al archivo de ondas crudas.
 - fs=128: frecuencia de muestreo del BIS. Hay 128 muestras por segundo.
 - escala_uv=0.0511: factor de conversión para transformar los valores enteros del archivo binario en microvoltios.
 
Devuelve:
 - df_eeg: DataFrame con columnas: tiempo_s, canal_1_raw, canal_2_raw, canal_1_uV, canal_2_uV.
           señal cruda de los dos canales en microvoltios


### Reconstrucción espectral del canal
Parámetros:
 - df_eeg: contiene el EEG crudo leído antes.
 - "canal_1_uV": se usa la señal del canal 1 en microvoltios.
 - fs=128: frecuencia de muestreo.
 - ventana_seg=2: cada espectro se calcula con ventanas de 2 segundos.
 - paso_seg=1: la ventana avanza cada segundo.
 - fmin=0.5: frecuencia mínima representada.
 - fmax=30.0: frecuencia máxima representada.
 - paso_freq=0.5: resolución frecuencial de 0.5 Hz.
 - modo="densidad": se obtiene densidad espectral de potencia.
 - tiempo_referencia="centro": cada ventana se etiqueta por su centro temporal.
 
Devuelve:
 - df_dsa_canal1: matriz tiempo-frecuencia del canal 1.
 - frecuencias_c1: array con las frecuencias usadas, desde 0.5 hasta 30 Hz.
 
 
### Selección de columnas de frecuencia (cols_freq)
El DataFrame df_dsa_canal1 tiene una columna temporal llamada tiempo_s y luego muchas columnas de frecuencia:
    tiempo_s | 0.5 | 1.0 | 1.5 | ... | 30.0
    
Crear una lista con todas las columnas excepto tiempo_s


### Crear estructura para la DSA media (df_dsa_media)
 - Se hace una copia del df_dsa_canal1 para la conservación de la estructura: tiempo y frecuencias
 - Se usa de base para sobreescribir con la media de los dos canales
 

### Media de potencia de los dos canales (pot_media)
 - Extraer las columnas de frecuencia de cada canal:
     - df_dsa_canal1[cols_freq]
     - df_dsa_canal2[cols_freq]
 - Convertir a array numérico: 
     - .to_numpy(dtype=float)
 - Sumar ambos canales y dividir entre dos
Devuelve: pot_media = matriz de numpy tiempo x frecuencia


### Convertir frecuencias a número (frecuencias_float)
 - El cabecero estaba escrito en formato texto y se pasa a numérico
Devuelve: array numérico con las frecuencias


### Cálculo de SEF y MEF propios
Parámetros:
 - potencia=pot_media: matriz de potencias en función del tiempo en escala lineal.
 - frecuencias=frecuencias_float: frecuencias correspondientes a cada columna.
 - percentil_sef=0.95: SEF es la frecuencia por debajo de la cual se acumula el 95 % de la potencia.
 - percentil_mef=0.50: MEF es la frecuencia por debajo de la cual se acumula el 50 % de la potencia.

Devuelve:
 - sef_eeg: array con el valor del parámetro por segundo 
 - mef_eeg: array con el valor valor del parámetro por segundo
 
Como la reconstrucción con ventanas de 2 segundos requiere que se añada una fila inicial NaN, como queremos que el SEF y el MEF tenga la misma longitud que el SEF, también se le añade un NaN al principio.
 - np.r_[np.nan, sef_eeg]: concatena un NaN inicial con el vector sef_eeg.
 - pd.Series(...): lo convierte en una serie de pandas.
 - name="SEF08": le da nombre compatible con tu función de plot.

Devuelve:
 - mef_eeg_plot: serie con el mef propio preparado para meter al plot
 - sef_eeg_plot: serie con el sef propio preparado para meter al plot
 
 
### Conversión de potencia a dB (df_dsa_media[cols_freq])
Se transforma la potencia media lineal a decibelios:
 - pot_media: matriz de potencia/densidad en escala lineal.
 - 1e-12: valor muy pequeño para evitar log10(0).
 - ref_uv_rms ** 2: potencia.
 - 10 * np.log10(...): conversión a decibelios.
 
El resultado se guarda en el dF con la media de potencias y sobreescribiendo el contenido de las columnas.

### Preparar el .spa temporalmente (fun_dsa.obtener_hora_inicio_desde_spa)
Parámetros:
 - df_spa_unilat: dF con las variables procesadas

Devuelve:
 - df_spa_unilat: el DataFrame .spa preparado, con la columna Time en formato datetime y redondeada al segundo.
 - hora_inicio: hora inicial.

Se limpia el .spa para quedarnos con las filas que sí pueden alinearse temporalmente. Después, la DSA mantiene su propio eje temporal continuo con tiempo_eeg en posteriores pasos.
 

### Adaptar DSA reconstruida al tiempo real (fun_dsa.adaptar_dsa_reconstruida_para_plot)
Parámetros:
 - df_dsa=df_dsa_media: dF con la DSA reconstruida en dB.
 - frecuencias=frecuencias_c1: frecuencias de la matriz.********************** POR QUÉ NO SE USA FRECUENCIAS_FLOAT
 - hora_inicio=hora_inicio: hora inicial.
 - insertar_fila_inicial_nan=True: añade una fila inicial con NaN para compensar que la primera ventana centrada cae en el segundo 1. ************************ EXPLICACIÓN EN EL CUADERNO

Devuelve:
 - tiempo_eeg: serie temporal real, con fechas y horas. ************************** SERIE DE PANDAS?
 - dsa_eeg: dF matriz DSA en dB, ya sin columna tiempo_s. ********************************** DF?

 
### Alinear .spa con la DSA reconstruida (fun_dsa.alinear_spa_con_tiempo)
Alinear las variables del .spa con los tiempos de la DSA reconstruida.

Se crea una tabla con todos los tiempos de la DSA. Si en el .spa falta información para algún segundo, ese segundo no desaparece: queda en la DSA, pero con valores del .spa como NaN. 
 - Por eso el merge se hace con how="left": conserva todos los tiempos de la DSA aunque el .spa no tenga datos para alguno de ellos.

Parámetros:
 - tiempo=tiempo_eeg: serie temporal. Fecha y hora de cada registro. Un registro cada segundo
 - df_spa=df_spa_unilat: el DataFrame .spa preparado, con la columna Time en formato datetime y redondeada al segundo.

Devuelve:
 - df_merge_plot : DataFrame con una fila por cada instante de tiempo_eeg y añade las variables del .spa que coinciden en ese segundo.
 
Usarán variables del SPA como:  SQI10, TOTPOW08, SEF08 en la misma línea temporal que la DSA.


### Sustituir SEF y MEF por los propios
 - mef_eeg_plot.values: recoge los valores mef de la serie
 - sef_eeg_plot.values: recoge los valores sef de la serie
 
Devuelve:
 - df_merge_plot["MEDFRQ08"]: como el spa tenía valores SEF y MEF procesados, los cambia por los calculados
 
 
### Calcular máscara total (fun_dsa.preparar_dsa_con_mask)
La máscara marca como no válidos los segundos donde se cumplen criterios

Parámetros:
 - tiempo=tiempo_eeg: serie temporal. Fecha y hora de cada registro. Un registro cada segundo
 - dsa=dsa_eeg: matriz DSA en dB, ya sin columna tiempo_s.
 - df_merge=df_merge_plot: df_merge_plot : DataFrame con una fila por cada instante de tiempo_eeg y añade las variables del .spa que coinciden en ese segundo.
 - umbral_sqi=15: criterio de invalidez de fila
 - umbral_ceros=0.9: criterio de invalidez de fila

Devuelve:
 - dsa preparada para plot: no hace falta así que se suele poner un _
 - mask_total: serie booleana (False=fila válida, True=fila no válida)
 
 
### Crear máscara común
Utiliza:
 - mask_total.copy(): copia de la máscara total para usarla como máscara común en todas las matrices.

Devuelve:
 - mask_comun: evitamos aplicar máscaras distintas en diferentes puntos del flujo.
 
 
### Aplicar máscara común a la DSA original f_a

### Preparar escala de color para la DSA reconstruida (fun_dsa.preparar_escala_color_dsa)
Parámetros:
 - dsa_eeg_directa_plot:
 - gamma: parámetro que modifica la distribución de colores para mejorar el parecido con la DSA del BIS. No cambia los datos, solo la representación.

Devuelve:
 - matriz_eeg: matriz NumPy para imshow.
 - vmin_eeg: valor mínimo de la escala de color.
 - vmax_eeg: valor máximo de la escala de color.
 - norm_eeg: normalización de color.
 - cmap_eeg: colormap tipo BIS.